Loading DataSet

In [106]:
import pandas as pd

df = pd.read_excel(r"D:\Crop_yeild\Crop_Yield_Messy_Dataset_500.xlsx")

Dropping duplicate value and stripping columns

In [107]:
df.columns = df.columns.str.strip().str.lower()
df = df.drop_duplicates()

Converting negative values in nan

In [108]:
import numpy as np

num_col = df.select_dtypes(include="number").columns
df[num_col] = df[num_col].mask(df[num_col] < 0, np.nan)

Stripping categorical columns and solving unique value related issue's

In [109]:
cat_col = df.select_dtypes(include=["string", "object", "category"]).columns

for col in cat_col:
    df[col] = df[col].str.strip().str.lower()

df["state"] = df["state"].replace({
    "wb" : "west bengal"
})

Split the data into train and test data

In [110]:
from sklearn.model_selection import train_test_split

X = df.drop("yield_ton_per_hectare", axis=1)
y = df["yield_ton_per_hectare"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Importing necessary librarie's

In [111]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

Creating pipelines

In [112]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", MinMaxScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

Separating numerical and categorical column's

In [113]:
numeric_col = X_train.select_dtypes(include="number").columns
categorical_col = X_train.select_dtypes(include=["string", "category", "object"]).columns

Applying ColumnTransformer

In [114]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numeric_col),
    ("cat", categorical_pipeline, categorical_col)
])

Applying LinearRegression Model

In [115]:
from sklearn.linear_model import LinearRegression

model_LiR = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

model_LiR.fit(X_train, y_train)
y_pred_LiR = model_LiR.predict(X_test)

Evaluation

In [116]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("MAE :", round(mean_absolute_error(y_test, y_pred_LiR),4))
print("MSE :", round(mean_squared_error(y_test, y_pred_LiR), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_LiR)), 4))
print("R² Score:", round(r2_score(y_test, y_pred_LiR), 4))

MAE : 0.5079
MSE : 0.3302
RMSE: 0.5746
R² Score: 0.6858


Applying RandomForestRegressor Model

In [117]:
from sklearn.ensemble import RandomForestRegressor

model_RFR = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_estimators=100))
])

model_RFR.fit(X_train, y_train)
y_pred_RFR = model_RFR.predict(X_test)

Evaluation

In [118]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("MAE :", round(mean_absolute_error(y_test, y_pred_RFR), 4))
print("MSE :", round(mean_squared_error(y_test, y_pred_RFR), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_RFR)), 4))
print("R² Score:", round(r2_score(y_test, y_pred_RFR), 4))

MAE : 0.5499
MSE : 0.4125
RMSE: 0.6422
R² Score: 0.6075


Applying XGBRegressor Model

In [119]:
from xgboost import XGBRegressor

model_XGBR = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(random_state=42, n_estimators=100, max_depth=6, learning_rate=0.1))
])

model_XGBR.fit(X_train, y_train)
y_pred_XGBR = model_XGBR.predict(X_test)

Evaluation

In [120]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("MAE :", round(mean_absolute_error(y_test, y_pred_XGBR), 4))
print("MSE :", round(mean_squared_error(y_test, y_pred_XGBR), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_XGBR)), 4))
print("R² Score:", round(r2_score(y_test, y_pred_XGBR), 4))

MAE : 0.5453
MSE : 0.4441
RMSE: 0.6664
R² Score: 0.5774


Declaring r2 scores as a variable

In [121]:
r2_LiR = r2_score(y_test, y_pred_LiR)
r2_RFR = r2_score(y_test, y_pred_RFR)
r2_XGBR = r2_score(y_test, y_pred_XGBR)

Converting into data frame

In [122]:
import pandas as pd

results = {
    "Algorithm": [
        "Linear Regression",
        "Random Forest Regressor",
        "XGBoost Regressor"
    ],
    "R² Score": [
        r2_LiR,
        r2_RFR,
        r2_XGBR
    ]
}

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="R² Score", ascending=False)

print(results_df)

                 Algorithm  R² Score
0        Linear Regression  0.685810
1  Random Forest Regressor  0.607504
2        XGBoost Regressor  0.577360


Cross Checking 

In [123]:
from sklearn.model_selection import cross_val_score
models = {
    "Linear Regression": model_LiR,
    "Random Forest": model_RFR,
    "XGBoost": model_XGBR
}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="r2"
    )

    print(f"{name}:")
    print("Scores:", scores)
    print("Average R²:", round(scores.mean(), 4))

Linear Regression:
Scores: [0.72944807 0.68146732 0.59357534 0.7269836  0.65056303]
Average R²: 0.6764
Random Forest:
Scores: [0.67800645 0.58964788 0.59052919 0.69828304 0.60551194]
Average R²: 0.6324
XGBoost:
Scores: [0.66299689 0.56586984 0.55320433 0.64857065 0.58919046]
Average R²: 0.604


Feature importance checking

In [124]:
feature_importance = pd.DataFrame({
    "Feature": model_LiR.named_steps["preprocessor"].get_feature_names_out(),
    "Coefficient": model_LiR.named_steps["model"].coef_
})

feature_importance.sort_values(by="Coefficient", ascending=False)

,Feature,Coefficient
1,num__rainfall_mm,2.934765
4,num__nitrogen,0.776766
6,num__potassium,0.608031
5,num__phosphorus,0.422041
3,num__humidity,0.345090
20,cat__crop_cotton,0.077609
23,cat__crop_potato,0.065818
15,cat__district_ludhiana,0.060493
12,cat__state_punjab,0.060493
33,cat__soil_type_red,0.047678


Declaration of final model

In [125]:
print("Final Model: Linear Regression")

Final Model: Linear Regression


Saving the model

In [126]:
import joblib

joblib.dump(model_LiR, "crop_yield_prediction_model.pkl")

print("Model saved successfully!")

Model saved successfully!
